# LeetCode 3850 — Count Sequences to K bằng Python

- Link: https://leetcode.com/problems/count-sequences-to-k/
- Độ khó: **Hard**
- Chủ đề: Phân tích thừa số nguyên tố, quy hoạch động trên vector số mũ, đệ quy có nhớ với phân số tối giản, meet-in-the-middle

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countSequences(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: vì `nums[i] ∈ [1, 6]`, mọi giá trị trung gian có dạng `2^a · 3^b · 5^c` với số mũ **nguyên (có thể âm)** ⇒ nhân/chia trở thành **cộng/trừ vector số mũ**.
2. Kiểm tra nhanh: nếu `k` có thừa số nguyên tố khác `2, 3, 5` thì đáp án `0`.
3. Viết DP "đếm số cách gán dấu `+1 / −1 / 0`" trên vector số mũ bằng `dict`/`Counter`.
4. So sánh với đệ quy có nhớ trên **phân số tối giản** `(p, q)` và với **meet-in-the-middle** (`3^{n/2}` mỗi nửa).
5. Kiểm thử với vét cạn `3^n` dùng `fractions.Fraction`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Bắt đầu với `val = 1`, duyệt `nums` từ trái sang phải. Tại mỗi phần tử chọn **đúng một** thao tác: nhân `val` với `nums[i]`, chia `val` cho `nums[i]` (chia **hữu tỉ chính xác**), hoặc giữ nguyên. Đếm số **dãy lựa chọn** khác nhau làm `val` cuối cùng **bằng đúng** `k`.

**Ví dụ 1:** `nums = [2,3,2], k = 6` → `2` (nhân, nhân, giữ; hoặc giữ, nhân, nhân).

**Ví dụ 2:** `nums = [4,6,3], k = 2` → `2` (`4/6·3 = 2`; `6/3 = 2`).

**Ví dụ 3:** `nums = [1,5], k = 1` → `3` (với `1`: nhân, chia hay giữ đều cho 1 — là ba dãy **khác nhau**).

**Ràng buộc:** `1 ≤ n ≤ 19`, `1 ≤ nums[i] ≤ 6`, `1 ≤ k ≤ 10^15`.

In [1]:
from typing import List
from fractions import Fraction
from functools import cache
from itertools import product
from collections import Counter
from math import gcd
import random, time

examples = [([2, 3, 2], 6, 2), ([4, 6, 3], 2, 2), ([1, 5], 1, 3)]
for nums, k, exp in examples:
    print(nums, "k =", k, "-> kỳ vọng", exp)

[2, 3, 2] k = 6 -> kỳ vọng 2
[4, 6, 3] k = 2 -> kỳ vọng 2
[1, 5] k = 1 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. `fractions.Fraction` — số hữu tỉ chính xác

`Fraction(4, 6)` tự rút gọn thành `2/3`. Phép nhân/chia giữ chính xác (không sai số dấu phẩy động).

In [2]:
v = Fraction(1)
v = v * 4 / 6 * 3
print(v, v == 2, Fraction(2, 4))

2 True 1/2


### 3.2. Phân tích thừa số nguyên tố thành vector số mũ

Các số `1..6` chỉ chứa thừa số `2, 3, 5`. Viết `x = 2^a 3^b 5^c` ⇒ vector `(a, b, c)`. Nhân ⇒ cộng vector; chia ⇒ trừ vector.

In [3]:
def exps(x, primes=(2, 3, 5)):
    res = []
    for p in primes:
        e = 0
        while x % p == 0:
            x //= p; e += 1
        res.append(e)
    return tuple(res), x            # x còn lại = phần không phân tích được

for x in range(1, 7):
    print(x, exps(x)[0])
print("k = 360 ->", exps(360), "| k = 14 ->", exps(14))

1 (0, 0, 0)
2 (1, 0, 0)
3 (0, 1, 0)
4 (2, 0, 0)
5 (0, 0, 1)
6 (1, 1, 0)
k = 360 -> ((3, 2, 1), 1) | k = 14 -> ((1, 0, 0), 7)


### 3.3. Đếm bằng `Counter` làm bảng DP

`dp` là `Counter` ánh xạ **trạng thái → số cách**. Mỗi bước, tạo `Counter` mới từ ba chuyển trạng thái. Trạng thái nào không xuất hiện coi như 0 cách.

In [4]:
dp = Counter({0: 1})
for x in [1, 2]:
    nd = Counter()
    for s, c in dp.items():
        nd[s + x] += c; nd[s - x] += c; nd[s] += c
    dp = nd
print(dict(dp))      # số cách đạt từng tổng khi gán +x, -x, 0

{3: 1, -1: 2, 1: 2, -3: 1, 2: 1, -2: 1, 0: 1}


### 3.4. Meet-in-the-middle

Chia dãy làm hai nửa; liệt kê mọi kết quả của nửa trái (`3^{n/2}`) vào `Counter`, rồi với mỗi kết quả `v` của nửa phải tra số cách nửa trái đạt `target − v`. Từ `3^{19} ≈ 1,16·10^9` xuống `2 · 3^{10} ≈ 1,2·10^5`.

In [5]:
print("3^19 =", 3**19, "| 3^10 + 3^9 =", 3**10 + 3**9)

3^19 = 1162261467 | 3^10 + 3^9 = 78732


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `nums = [4, 6, 3]`, `k = 2`. Vector: `4 → (2,0,0)`, `6 → (1,1,0)`, `3 → (0,1,0)`, mục tiêu `2 → (1,0,0)`.

Ta cần chọn hệ số `s_i ∈ {+1, −1, 0}` sao cho `Σ s_i · v_i = (1, 0, 0)`:

- `(+1)(2,0,0) + (−1)(1,1,0) + (+1)(0,1,0) = (1, 0, 0)` ✔ — tức `4 / 6 · 3`.
- `0·(2,0,0) + (+1)(1,1,0) + (−1)(0,1,0) = (1, 0, 0)` ✔ — tức `6 / 3`.

Bài toán phân số trở thành bài toán **đếm tổ hợp hệ số trên vector số nguyên**. Riêng `nums[i] = 1` có vector `(0,0,0)` ⇒ cả ba lựa chọn đều không đổi giá trị ⇒ nhân số cách lên **3**.

## 5. Cách 1 — Brute Force: thử mọi `3^n` dãy lựa chọn

Dùng `Fraction` để tính chính xác. Chỉ dùng khi `n ≤ 10` hoặc 11.

In [6]:
def brute(nums, k):
    cnt = 0
    for ops in product((0, 1, 2), repeat=len(nums)):
        v = Fraction(1)
        for x, op in zip(nums, ops):
            if op == 1:
                v *= x
            elif op == 2:
                v /= x
        cnt += v == k
    return cnt

for nums, k, exp in examples:
    print(brute(nums, k), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


**Nút thắt:** `3^{19} ≈ 1,16·10^9` dãy. Nhưng nhiều dãy dẫn tới **cùng giá trị** trung gian ⇒ gộp theo giá trị (DP).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Lọc `k`:** phân tích `k = 2^A 3^B 5^C · rest`. Nếu `rest ≠ 1` ⇒ trả `0`.

**Bước 2 — Trạng thái nhỏ:** sau `i` bước, vector số mũ `(a, b, c)` có `|a| ≤ 2i`, `|b| ≤ i`, `|c| ≤ i`. Với `n = 19`: tối đa `77 · 39 · 39 ≈ 1,2·10^5` trạng thái — nhỏ hơn rất nhiều `3^{19}`.

**Bước 3 — DP:** `dp_{i+1}[s + v_i] += dp_i[s]`, `dp_{i+1}[s − v_i] += dp_i[s]`, `dp_{i+1}[s] += dp_i[s]`. Đáp án `dp_n[(A, B, C)]`.

**Cắt tỉa (tùy chọn):** nếu từ trạng thái hiện tại, phần còn lại không thể bù đủ khoảng cách tới đích (tổng `|v|` còn lại theo từng thành phần nhỏ hơn khoảng cách) thì bỏ trạng thái.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy có nhớ trên phân số tối giản `(i, p, q)` (rút gọn bằng `gcd`).
- **Cách 3:** DP trên vector số mũ `(a, b, c)` (lời giải nộp chính).
- **Cách 4:** meet-in-the-middle trên vector số mũ.

### 7.1. Cách 2 — Đệ quy có nhớ với phân số `(p, q)`

In [7]:
def memo_fraction(nums, k):
    n = len(nums)

    @cache
    def dfs(i, p, q):
        if i == n:
            return 1 if p == k and q == 1 else 0
        x = nums[i]
        res = dfs(i + 1, p, q)
        g = gcd(p * x, q)
        res += dfs(i + 1, p * x // g, q // g)
        g = gcd(p, q * x)
        res += dfs(i + 1, p // g, q * x // g)
        return res

    ans = dfs(0, 1, 1)
    dfs.cache_clear()
    return ans

for nums, k, exp in examples:
    print(memo_fraction(nums, k), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** số trạng thái bằng số phân số khác nhau đạt được — tương đương số vector số mũ (≈ `10^5`), nhân hằng số của `gcd`. Bộ nhớ cache cùng cỡ. Phải `cache_clear()` để không giữ trạng thái giữa các lần gọi.

### 7.2. Cách 3 — DP trên vector số mũ (lời giải chính)

**Giải thích từng đoạn:**

1. Phân tích `k`; nếu còn thừa số khác `2, 3, 5` ⇒ `0`.
2. `dp = Counter({(0,0,0): 1})`.
3. Với mỗi `x`, `v = exps(x)`. Nếu `v = (0,0,0)` (tức `x = 1`) thì chỉ cần nhân mọi giá trị với 3. Ngược lại tạo `Counter` mới với ba chuyển trạng thái.
4. Trả `dp[(A, B, C)]`.

In [8]:
def exponent_dp(nums, k):
    target, rest = exps(k)
    if rest != 1:
        return 0
    dp = Counter({(0, 0, 0): 1})
    for x in nums:
        v = exps(x)[0]
        if v == (0, 0, 0):
            for s in dp:
                dp[s] *= 3
            continue
        nd = Counter()
        for (a, b, c), cnt in dp.items():
            nd[(a, b, c)] += cnt
            nd[(a + v[0], b + v[1], c + v[2])] += cnt
            nd[(a - v[0], b - v[1], c - v[2])] += cnt
        dp = nd
    return dp[target]

for nums, k, exp in examples:
    print(exponent_dp(nums, k), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp Cách 3:** `O(n · Smax)` với `Smax ≈ 1,2·10^5` trạng thái ⇒ khoảng `2·10^6` cập nhật `Counter` — chạy trong vài giây ở ca xấu nhất (thường nhanh hơn nhiều vì số trạng thái thực tế nhỏ hơn).

### 7.3. Cách 4 — Meet-in-the-middle

Liệt kê mọi vector đạt được của nửa trái (`3^{⌈n/2⌉}`) vào `Counter`; với mỗi vector `w` của nửa phải cộng `left[target − w]`.

In [9]:
def all_sums(part):
    res = Counter({(0, 0, 0): 1})
    for x in part:
        v = exps(x)[0]
        nd = Counter()
        for (a, b, c), cnt in res.items():
            nd[(a, b, c)] += cnt
            nd[(a + v[0], b + v[1], c + v[2])] += cnt
            nd[(a - v[0], b - v[1], c - v[2])] += cnt
        res = nd
    return res

def meet_in_middle(nums, k):
    target, rest = exps(k)
    if rest != 1:
        return 0
    h = len(nums) // 2
    left, right = all_sums(nums[:h]), all_sums(nums[h:])
    A, B, C = target
    return sum(cnt * left.get((A - a, B - b, C - c), 0) for (a, b, c), cnt in right.items())

for nums, k, exp in examples:
    print(meet_in_middle(nums, k), "kỳ vọng", exp)

2 kỳ vọng 2
2 kỳ vọng 2
3 kỳ vọng 3


In [10]:
class SolutionMITM:
    def countSequences(self, nums: List[int], k: int) -> int:
        return meet_in_middle(nums, k)

print([SolutionMITM().countSequences(n_, k) for n_, k, _ in examples])

[2, 2, 3]


**Độ phức tạp Cách 4:** mỗi nửa có tối đa `3^{10}` vector (thực tế ít hơn vì gộp) ⇒ rất nhanh, bộ nhớ `O(3^{n/2})`.

Lưu ý: `all_sums` ở đây đã **gộp** trạng thái trùng (là DP trên nửa dãy), nên thực chất là "DP hai nửa + ghép" — mạnh hơn liệt kê thuần.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 (`nums = [4,6,3]`, `k = 2`, đích `(1,0,0)`).

| bước | x | v | số trạng thái | vài trạng thái (vector: số cách) |
|---|---|---|---|---|
| 0 | – | – | 1 | (0,0,0): 1 |
| 1 | 4 | (2,0,0) | 3 | (0,0,0): 1, (2,0,0): 1, (−2,0,0): 1 |
| 2 | 6 | (1,1,0) | 9 | … |
| 3 | 3 | (0,1,0) | … | (1,0,0): **2** |

In [11]:
def trace_dp(nums, k):
    target, rest = exps(k)
    print("đích:", target, "| phần dư:", rest)
    dp = Counter({(0, 0, 0): 1})
    for i, x in enumerate(nums, 1):
        v = exps(x)[0]
        nd = Counter()
        for (a, b, c), cnt in dp.items():
            nd[(a, b, c)] += cnt
            nd[(a + v[0], b + v[1], c + v[2])] += cnt
            nd[(a - v[0], b - v[1], c - v[2])] += cnt
        dp = nd
        print(f"bước {i}: x={x}, v={v}, {len(dp)} trạng thái, số cách tới đích hiện tại = {dp[target]}")
    print("trạng thái cuối:", dict(sorted(dp.items())))
    return dp[target]

trace_dp([4, 6, 3], 2)

đích: (1, 0, 0) | phần dư: 1
bước 1: x=4, v=(2, 0, 0), 3 trạng thái, số cách tới đích hiện tại = 0
bước 2: x=6, v=(1, 1, 0), 9 trạng thái, số cách tới đích hiện tại = 0
bước 3: x=3, v=(0, 1, 0), 25 trạng thái, số cách tới đích hiện tại = 2
trạng thái cuối: {(-3, -2, 0): 1, (-3, -1, 0): 1, (-3, 0, 0): 1, (-2, -1, 0): 1, (-2, 0, 0): 1, (-2, 1, 0): 1, (-1, -2, 0): 1, (-1, -1, 0): 1, (-1, 0, 0): 2, (-1, 1, 0): 1, (-1, 2, 0): 1, (0, -1, 0): 1, (0, 0, 0): 1, (0, 1, 0): 1, (1, -2, 0): 1, (1, -1, 0): 1, (1, 0, 0): 2, (1, 1, 0): 1, (1, 2, 0): 1, (2, -1, 0): 1, (2, 0, 0): 1, (2, 1, 0): 1, (3, 0, 0): 1, (3, 1, 0): 1, (3, 2, 0): 1}


2

In [12]:
# Ví dụ 3: phần tử 1 nhân số cách lên 3
print(trace_dp([1, 5], 1), "| giải thích: với x = 1 cả ba lựa chọn đều giữ (0,0,0)")

đích: (0, 0, 0) | phần dư: 1
bước 1: x=1, v=(0, 0, 0), 1 trạng thái, số cách tới đích hiện tại = 3
bước 2: x=5, v=(0, 0, 1), 3 trạng thái, số cách tới đích hiện tại = 3
trạng thái cuối: {(0, 0, -1): 3, (0, 0, 0): 3, (0, 0, 1): 3}
3 | giải thích: với x = 1 cả ba lựa chọn đều giữ (0,0,0)


## 9. Code nộp LeetCode

In [13]:
from collections import Counter

class Solution:
    def countSequences(self, nums: List[int], k: int) -> int:
        def exps(x):
            e = []
            for p in (2, 3, 5):
                c = 0
                while x % p == 0:
                    x //= p
                    c += 1
                e.append(c)
            return tuple(e), x

        target, rest = exps(k)
        if rest != 1:
            return 0
        dp = Counter({(0, 0, 0): 1})
        for x in nums:
            (va, vb, vc), _ = exps(x)
            if va == vb == vc == 0:
                for s in dp:
                    dp[s] *= 3
                continue
            nd = Counter()
            for (a, b, c), cnt in dp.items():
                nd[(a, b, c)] += cnt
                nd[(a + va, b + vb, c + vc)] += cnt
                nd[(a - va, b - vb, c - vc)] += cnt
            dp = nd
        return dp[target]

sol = Solution()
for nums, k, exp in examples:
    got = sol.countSequences(nums, k)
    print(nums, k, "->", got, "✔" if got == exp else "✘")

[2, 3, 2] 6 -> 2 ✔
[4, 6, 3] 2 -> 2 ✔
[1, 5] 1 -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Đồng cấu:** ánh xạ `2^a 3^b 5^c ↦ (a, b, c)` là song ánh từ tập các số hữu tỉ dương chỉ chứa thừa số `2, 3, 5` sang `Z^3`, biến phép nhân thành phép cộng. Mọi giá trị trung gian thuộc tập này (bắt đầu từ `1`, nhân/chia cho số `≤ 6`).
2. **`k` ngoài tập ⇒ 0:** giá trị cuối luôn chỉ chứa thừa số `2, 3, 5`; nếu `k` có thừa số khác thì không thể bằng.
3. **DP đếm đúng:** mỗi dãy lựa chọn tương ứng một dãy hệ số `s_i ∈ {+1, −1, 0}`, và hai dãy lựa chọn khác nhau là hai dãy hệ số khác nhau (ba lựa chọn tương ứng ba hệ số) — **kể cả khi `x = 1`** (ba hệ số khác nhau nhưng cùng vector 0, nên được đếm 3 lần). `dp_i[s]` = số dãy hệ số độ dài `i` có tổng `s` (quy nạp theo `i`).

## 11. Phân tích Time / Space Complexity

Đặt `n ≤ 19`, `S` = số trạng thái vector (≤ `(4n+1)(2n+1)^2 ≈ 1,2·10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(3^n · n)` | `O(n)` |
| 2. Memo phân số | `O(n · S · log)` | `O(n · S)` |
| 3. DP vector | `O(n · S)` | `O(S)` |
| 4. Hai nửa + ghép | `O(n · S_{1/2} + S_{1/2})` | `O(S_{1/2})` |

## 12. Bộ test

In [14]:
def run_all(nums, k, with_brute=True):
    res = [memo_fraction(nums, k), exponent_dp(nums, k), meet_in_middle(nums, k), Solution().countSequences(nums, k)]
    if with_brute:
        res.append(brute(nums, k))
    return res

for nums, k, exp in examples:
    assert all(v == exp for v in run_all(nums, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([1], 1, 3),                 # ba lựa chọn đều cho 1
    ([1, 1, 1], 1, 27),
    ([6], 6, 1), ([6], 7, 0),    # 7 có thừa số nguyên tố khác
    ([2, 2], 1, 3),              # giữ-giữ, nhân-chia, chia-nhân
    ([5] * 6, 5**6, 1),
    ([3, 4], 10**15, 0),         # k quá lớn
]
for nums, k, exp in cases:
    r = run_all(nums, k)
    assert all(v == exp for v in r), (nums, k, r)
print("Ca biên: OK")

Ca biên: OK


In [16]:
# 12.3 phản ví dụ: bỏ qua "phần dư" của k (chỉ so số mũ 2, 3, 5) sẽ đếm nhầm
def wrong_ignore_rest(nums, k):
    target = exps(k)[0]
    dp = Counter({(0, 0, 0): 1})
    for x in nums:
        v = exps(x)[0]; nd = Counter()
        for (a, b, c), cnt in dp.items():
            nd[(a, b, c)] += cnt; nd[(a + v[0], b + v[1], c + v[2])] += cnt; nd[(a - v[0], b - v[1], c - v[2])] += cnt
        dp = nd
    return dp[target]

print("k = 14 (= 2·7): sai:", wrong_ignore_rest([2, 3], 14), "| đúng:", Solution().countSequences([2, 3], 14))

k = 14 (= 2·7): sai: 1 | đúng: 0


In [17]:
# 12.4 đối chiếu ngẫu nhiên (n nhỏ có vét cạn)
rng = random.Random(3850)
for _ in range(200):
    n = rng.randint(1, 8)
    nums = [rng.randint(1, 6) for _ in range(n)]
    if rng.random() < 0.6:
        k = 1
        for x in nums:
            r = rng.random()
            if r < 0.4: k *= x
        k = max(k, 1)
    else:
        k = rng.randint(1, 100)
    res = run_all(nums, k)
    assert len(set(res)) == 1, (nums, k, res)
print("200 test có vét cạn: OK")

200 test có vét cạn: OK


In [18]:
# 12.5 ca lớn nhất n = 19
rng = random.Random(9)
for nums, k in [([2, 3, 4, 5, 6] * 3 + [2, 3, 5, 6], 60), ([6] * 19, 1), ([4] * 19, 2**10), ([2, 3, 5, 4, 6, 1] * 3 + [5], 10**15)]:
    t0 = time.time(); a = Solution().countSequences(nums, k); t1 = time.time()
    b = meet_in_middle(nums, k)
    print(f"k={k}: đáp án {a}, khớp MITM {a == b}, thời gian {t1 - t0:.2f}s")

k=60: đáp án 3637056, khớp MITM True, thời gian 0.01s
k=1: đáp án 128996853, khớp MITM True, thời gian 0.00s
k=1024: đáp án 49366674, khớp MITM True, thời gian 0.00s
k=1000000000000000: đáp án 0, khớp MITM True, thời gian 0.01s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng số thực (`float`) cho phép chia

Sai số dấu phẩy động có thể làm `val == k` sai (ví dụ `4/6*3` có thể ra `1.9999999999999998`).

In [19]:
v = 1.0
for x, op in [(5, '/'), (3, '/'), (5, '*'), (3, '*')]:
    v = v / x if op == '/' else v * x
print("float:", v, "== 1?", v == 1)
v = 1.0
for x in [6] * 19: v /= x
for x in [6] * 19: v *= x
print("chia rồi nhân 19 lần với 6:", repr(v), "== 1?", v == 1.0)

float: 1.0 == 1? True
chia rồi nhân 19 lần với 6: 0.9999999999999996 == 1? False


**Sửa:** dùng `Fraction`, phân số `(p, q)` tối giản bằng `gcd`, hoặc vector số mũ.

### Lỗi 2 — Coi ba lựa chọn với `x = 1` là một (đếm theo **giá trị**, không theo dãy lựa chọn)

Đề đếm **dãy lựa chọn**; với `x = 1` cả ba lựa chọn khác nhau dù cho cùng giá trị.

In [20]:
def wrong_dedupe_one(nums, k):
    dp = Counter({(0, 0, 0): 1}); target = exps(k)[0]
    for x in nums:
        v = exps(x)[0]; nd = Counter()
        for s, cnt in dp.items():
            for t in {s, tuple(a + b for a, b in zip(s, v)), tuple(a - b for a, b in zip(s, v))}:   # SAI: dùng set gộp
                nd[t] += cnt
        dp = nd
    return dp[target]

print("sai:", wrong_dedupe_one([1, 5], 1), "| đúng:", Solution().countSequences([1, 5], 1))

sai: 1 | đúng: 3


### Lỗi 3 — Không `cache_clear()` khi dùng `@cache` lồng trong hàm dùng biến ngoài

Nếu `@cache` đặt ở cấp module và hàm đọc `nums`, `k` từ biến toàn cục, lần gọi sau với dữ liệu khác sẽ dùng nhầm kết quả cũ. Hàm lồng (định nghĩa lại mỗi lần) hoặc `cache_clear()` sẽ tránh được.

In [21]:
STATE = {"nums": [], "k": 1}
@cache
def bad_dfs(i, p, q):
    nums, k = STATE["nums"], STATE["k"]
    if i == len(nums): return int(p == k and q == 1)
    x = nums[i]
    return bad_dfs(i + 1, p, q) + bad_dfs(i + 1, p * x // gcd(p * x, q), q // gcd(p * x, q)) + bad_dfs(i + 1, p // gcd(p, q * x), q * x // gcd(p, q * x))

STATE.update(nums=[2, 3, 2], k=6); a = bad_dfs(0, 1, 1)
STATE.update(nums=[2, 3, 2], k=3); b = bad_dfs(0, 1, 1)          # SAI: cache cũ trả lời cho k = 6
print("lần 2 (k=3): sai:", b, "| đúng:", Solution().countSequences([2, 3, 2], 3))

lần 2 (k=3): sai: 2 | đúng: 3


### Lỗi 4 — Quên rằng giá trị trung gian có thể là **phân số**

Nếu cấm chia khi không chia hết (chỉ cho phép chia nguyên), ta bỏ sót các dãy như `4 / 6 · 3` ở ví dụ 2.

In [22]:
def wrong_integer_only(nums, k):
    cnt = 0
    for ops in product((0, 1, 2), repeat=len(nums)):
        v, ok = 1, True
        for x, op in zip(nums, ops):
            if op == 1: v *= x
            elif op == 2:
                if v % x: ok = False; break       # SAI: cấm phân số
                v //= x
        cnt += ok and v == k
    return cnt

print("sai:", wrong_integer_only([4, 6, 3], 2), "| đúng:", Solution().countSequences([4, 6, 3], 2))

sai: 1 | đúng: 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhân/chia với các số nhỏ** ⇒ chuyển sang **vector số mũ nguyên tố**; phép nhân thành cộng.
- **Đếm số cách gán `+ / − / 0`** để đạt tổng ⇒ DP trên tổng (giống *Target Sum*, LC 494) — ở đây tổng là vector 3 chiều.
- `n ≤ 20` và `3^n` quá lớn ⇒ cân nhắc **meet-in-the-middle** hoặc DP gộp trạng thái.
- Luôn kiểm tra sớm điều kiện "không thể" (thừa số lạ trong `k`).

## 15. Biến thể của bài

1. **Chỉ nhân hoặc giữ** (không chia): số mũ chỉ tăng ⇒ DP nhỏ hơn, cắt tỉa khi vượt đích.
2. **`nums[i]` tới 100**: thêm các số nguyên tố `≤ 97` ⇒ vector nhiều chiều, dùng `dict` vẫn được nếu `n` nhỏ.
3. **Đếm số giá trị phân biệt** đạt được: `len(dp)` cuối cùng.
4. **Target Sum (LC 494)**: trường hợp một chiều không có lựa chọn "giữ".

In [23]:
# Biến thể 3: số giá trị hữu tỉ phân biệt đạt được
def distinct_values(nums):
    dp = {(0, 0, 0)}
    for x in nums:
        v = exps(x)[0]
        dp = {t for s in dp for t in (s, tuple(a + b for a, b in zip(s, v)), tuple(a - b for a, b in zip(s, v)))}
    return len(dp)

def brute_distinct(nums):
    vals = set()
    for ops in product((0, 1, 2), repeat=len(nums)):
        v = Fraction(1)
        for x, op in zip(nums, ops):
            v = v * x if op == 1 else (v / x if op == 2 else v)
        vals.add(v)
    return len(vals)

for nums in ([2, 3, 2], [4, 6, 3], [1, 5], [6, 6, 2]):
    assert distinct_values(nums) == brute_distinct(nums)
print("Biến thể 3 khớp vét cạn:", [distinct_values(n_) for n_ in ([2, 3, 2], [4, 6, 3], [1, 5], [6, 6, 2])])

Biến thể 3 khớp vét cạn: [15, 25, 3, 15]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `3^n` dãy với `Fraction` | `O(3^n n)` | `O(n)` | Kiểm thử, `n ≤ 10` |
| 2. Memo phân số | `(i, p, q)` tối giản | `O(n S log)` | `O(n S)` | Khi không nghĩ ra vector số mũ |
| 3. DP vector | `Counter` trên `(a, b, c)` | `O(n S)` | `O(S)` | **Lời giải chuẩn** |
| 4. Hai nửa + ghép | Meet-in-the-middle | `O(n S_{1/2})` | `O(S_{1/2})` | Khi `S` lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countSequences([2, 2, 2], 2)` = ?

**Bài 2 (tính tay).** `countSequences([2, 4], 2)` — liệt kê các dãy hệ số trên số mũ của 2.

**Bài 3 (điền TODO).** Viết `target_sum_count(nums, target)` (LC 494, mỗi phần tử `+` hoặc `−`) bằng DP `Counter`.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với **cắt tỉa**: bỏ trạng thái mà phần còn lại không thể bù được khoảng cách tới đích (theo từng thành phần số mũ).

**Bài 5 (mini challenge).** Mở rộng cho `nums[i] ≤ 30` (thêm các số nguyên tố 7, 11, 13, …, 29). Viết hàm tổng quát và kiểm chứng vét cạn.

In [24]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def target_sum_count(nums, target):
    # TODO: dp = Counter({0: 1}); mỗi x: nd[s + x] += c; nd[s - x] += c
    pass

In [25]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_general(nums, k):
    # TODO: primes = các số nguyên tố <= max(nums); vector số mũ theo primes; DP Counter
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Số mũ của 2: mỗi phần tử `+1/−1/0`, cần tổng `1`. (2) Đếm bộ `(s1, s2, s3)` tổng 1: một `+1` hai `0` (3 cách) hoặc hai `+1` một `−1` (3 cách). (3) Đáp án `6`.

**Bài 2.** (1) `2 → 1`, `4 → 2`. (2) Cần `s1·1 + s2·2 = 1`: `(1, 0)`, `(−1, 1)`. (3) Đáp án `2`.

**Bài 3.** (1) Hai chuyển trạng thái. (2) Đáp án `dp[target]`. (3) LC 494 ví dụ `[1,1,1,1,1], 3 → 5`.

**Bài 4.** (1) Tính hậu tố tổng `|v|` theo từng thành phần. (2) Giữ trạng thái chỉ khi `|đích − s| ≤ hậu tố` ở mọi thành phần. (3) Kết quả không đổi, số trạng thái giảm.

**Bài 5.** (1) Phân tích theo danh sách số nguyên tố. (2) `k` còn dư ⇒ 0. (3) Vét cạn với `Fraction` khi `n ≤ 8`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().countSequences([2, 2, 2], 2) == 6 == brute([2, 2, 2], 2)
assert Solution().countSequences([2, 4], 2) == 2 == brute([2, 4], 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
# Bài 3
def target_sum_count_sol(nums, target):
    dp = Counter({0: 1})
    for x in nums:
        nd = Counter()
        for s, c in dp.items():
            nd[s + x] += c; nd[s - x] += c
        dp = nd
    return dp[target]

assert target_sum_count_sol([1, 1, 1, 1, 1], 3) == 5
assert target_sum_count_sol([1], 1) == 1
print("Bài 3 OK")

Bài 3 OK


In [28]:
# Bài 4: DP có cắt tỉa
def pruned_dp(nums, k):
    target, rest = exps(k)
    if rest != 1: return 0, 0
    vs = [exps(x)[0] for x in nums]
    n = len(nums)
    suf = [(0, 0, 0)] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = tuple(a + b for a, b in zip(suf[i + 1], vs[i]))
    dp = Counter({(0, 0, 0): 1}); peak = 1
    for i, v in enumerate(vs):
        nd = Counter()
        for s, c in dp.items():
            for t in (s, tuple(a + b for a, b in zip(s, v)), tuple(a - b for a, b in zip(s, v))):
                if all(abs(tt - g) <= r for tt, g, r in zip(t, target, suf[i + 1])):
                    nd[t] += c
        dp = nd; peak = max(peak, len(dp))
    return dp[target], peak

rng = random.Random(4)
for _ in range(100):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]; k = rng.choice([1, 2, 6, 12, 30, 45, 7])
    assert pruned_dp(nums, k)[0] == Solution().countSequences(nums, k)
print("Bài 4 OK; số trạng thái tối đa với n = 19 (có cắt tỉa):", pruned_dp([2, 3, 4, 5, 6] * 3 + [2, 3, 5, 6], 60)[1])

Bài 4 OK; số trạng thái tối đa với n = 19 (có cắt tỉa): 392


In [29]:
# Bài 5: tổng quát cho nums[i] <= 30
PR = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
def gexps(x):
    e = []
    for p in PR:
        c = 0
        while x % p == 0: x //= p; c += 1
        e.append(c)
    return tuple(e), x

def count_general_sol(nums, k):
    target, rest = gexps(k)
    if rest != 1: return 0
    dp = Counter({tuple([0] * len(PR)): 1})
    for x in nums:
        v = gexps(x)[0]; nd = Counter()
        for s, c in dp.items():
            nd[s] += c
            nd[tuple(a + b for a, b in zip(s, v))] += c
            nd[tuple(a - b for a, b in zip(s, v))] += c
        dp = nd
    return dp[target]

rng = random.Random(5)
for _ in range(100):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 7))]
    k = rng.choice([1, 7, 14, 21, 26, 30, 2 * 29])
    assert count_general_sol(nums, k) == brute(nums, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
target, rest = phân tích k theo (2, 3, 5)
if rest != 1: return 0
dp = Counter({(0,0,0): 1})
for x in nums:
    v = vector số mũ của x
    nd = Counter()
    for s, c in dp.items():
        nd[s] += c; nd[s + v] += c; nd[s - v] += c     # giữ / nhân / chia
    dp = nd
return dp[target]
```

**Checklist:**

- [ ] Không dùng `float`; dùng vector số mũ hoặc `Fraction`?
- [ ] Kiểm tra `k` có thừa số nguyên tố ngoài `2, 3, 5` ⇒ `0`?
- [ ] Ba lựa chọn với `x = 1` được đếm là ba dãy khác nhau?
- [ ] Không gộp trạng thái bằng `set` (phải cộng số cách)?
- [ ] Nếu dùng `@cache`, đã xóa cache giữa các test?